
# Build the vector store

Embeds the 62 addressable clauses (`data/addressable_clauses.csv`) with
`sentence-transformers/all-MiniLM-L6-v2` and stores them in a Chroma collection, so retrieval can find
the clauses most relevant to each Starhawk requirement **by meaning**, not just shared words.

**Output:** a Chroma collection in `data/chroma_db/` (gitignored; rebuilt from scratch on every run).

### Why long clauses are split

`all-MiniLM-L6-v2` reads at most **256 tokens** of any text and silently ignores the rest. 36 of the 62
clauses are longer than that (the longest is about 1,700 tokens), so most of their text would never
reach the embedding. Each clause is therefore split into **pieces of at most 200 tokens**, on sentence
boundaries, with one sentence of overlap. Every piece keeps its parent `chunk_id`, and `retrieve()`
returns each parent clause at most once.


In [1]:

import os, re, shutil, warnings, logging
from pathlib import Path

import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Resolve the repo root so this runs from the root or from notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "addressable_clauses.csv").exists())
CLAUSES_CSV = ROOT / "data" / "addressable_clauses.csv"
PERSIST_DIR = ROOT / "data" / "chroma_db"
COLLECTION = "addressable_clauses"
MODEL = "sentence-transformers/all-MiniLM-L6-v2"
PIECE_TOKENS = 200   # stays under the model's 256-token limit, with room for special tokens



## Load the clauses

Read with pandas so the embedded text is the clause itself. (`CSVLoader` prefixes every chunk with the
column name, `"text: ..."`, which ends up inside every embedding.) All metadata needed for a citation in
Mark's report format is kept: `swe_id`, `section`, `type` and `source_url`, not just `chunk_id` and
`source`. Chroma can't store empty values, so missing ones become `""`.


In [2]:

META = ["chunk_id", "swe_id", "section", "type", "source", "source_url"]

clauses = pd.read_csv(CLAUSES_CSV).fillna("")
clauses["section"] = clauses["section"].astype(str)
print(f"{len(clauses)} clauses from {CLAUSES_CSV.relative_to(ROOT)}")
clauses[META].head()


62 clauses from data/addressable_clauses.csv


,chunk_id,swe_id,section,type,source,source_url
0,npr-3.7.3-requirement,SWE-134,3.7.3,requirement,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
1,npr-3.10.1-requirement,SWE-147,3.10.1,requirement,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
2,npr-3.11.8-requirement,SWE-210,3.11.8,requirement,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
3,npr-3.11.8-note,SWE-210,3.11.8,note,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
4,npr-4.1.2-note,SWE-050,4.1.2,note,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...


## Split long clauses into pieces

In [3]:

st = SentenceTransformer(MODEL)
LIMIT = st.max_seq_length
n_tokens = lambda text: len(st.tokenizer(text)["input_ids"])
sentences = lambda text: [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def split_into_pieces(text, budget=PIECE_TOKENS, overlap=1):
    """Pack whole sentences into pieces of at most `budget` tokens, overlapping by `overlap` sentences."""
    pieces, current = [], []
    for sentence in sentences(text):
        if current and n_tokens(" ".join(current + [sentence])) > budget:
            pieces.append(" ".join(current))
            current = current[-overlap:] if overlap else []
        current.append(sentence)
    if current:
        pieces.append(" ".join(current))
    return pieces


documents = [
    Document(page_content=piece, metadata={**{k: row[k] for k in META}, "piece": i})
    for _, row in clauses.iterrows()
    for i, piece in enumerate(split_into_pieces(row["text"]))
]

too_long = clauses["text"].map(n_tokens) > LIMIT
print(f"model limit: {LIMIT} tokens | clauses over it: {too_long.sum()}/{len(clauses)}")
print(f"{len(documents)} pieces | longest piece: {max(n_tokens(d.page_content) for d in documents)} tokens")


model limit: 256 tokens | clauses over it: 36/62
164 pieces | longest piece: 200 tokens



## Build the collection

The folder is deleted first. `Chroma.from_documents` **adds** to an existing collection, so re-running
without clearing it stacks duplicate (and possibly stale) copies of every clause.


In [4]:

shutil.rmtree(PERSIST_DIR, ignore_errors=True)

embeddings = HuggingFaceEmbeddings(model_name=MODEL)
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=COLLECTION,
    persist_directory=str(PERSIST_DIR),
)
print(f"stored {vector_db._collection.count()} pieces in {PERSIST_DIR.relative_to(ROOT)}/")


stored 164 pieces in data/chroma_db/



## Retrieve

`retrieve()` searches the pieces and returns the top `k` **distinct** clauses, best first. To reuse the
store in another notebook, open it with `Chroma(collection_name=COLLECTION,
persist_directory=str(PERSIST_DIR), embedding_function=HuggingFaceEmbeddings(model_name=MODEL))`.


In [5]:

def retrieve(query, k=3, fetch=15):
    """Top-k distinct clauses for a query, as (chunk_id, distance, metadata). Lower distance = closer."""
    results, seen = [], set()
    for doc, distance in vector_db.similarity_search_with_score(query, k=fetch):
        cid = doc.metadata["chunk_id"]
        if cid not in seen:
            seen.add(cid)
            results.append((cid, round(distance, 3), doc.metadata))
        if len(results) == k:
            break
    return results


reqs = pd.read_csv(ROOT / "data" / "srs_requirements.csv")
for rid in ["SFMC-REQ-015", "SFMC-REQ-016", "SFMC-REQ-029"]:
    text = reqs.loc[reqs["req_id"] == rid, "text"].item()
    print(f"{rid}: {text[:90]}")
    for cid, dist, meta in retrieve(text):
        print(f"    {dist:5}  {cid:28} {meta['type']}")


SFMC-REQ-015: The SFMC shall never permit the pilot to fire a weapon at a friendly spacecraft.
    1.609  npr-3.7.3-requirement        requirement
     1.62  swehb-3.2.1.4-guidance       guidance
    1.636  swehb-3.2.2.3-guidance       guidance
SFMC-REQ-016: The targeting system shall be extremely accurate.
    1.347  swehb-6.1-guidance           guidance
    1.426  npr-3.11.8-note              note
    1.441  swehb-3.2.2.3-guidance       guidance
SFMC-REQ-029: For each recorded fault, the mission fault log shall contain: - fault identifier, - affect
    1.003  swehb-3.2.1.9-guidance       guidance
    1.015  swehb-6.1-guidance           guidance
    1.115  swehb-3.2.1.13-guidance      guidance



## Checks

1. Structure: every piece fits the model, every clause is stored, citation metadata is complete.
2. **Heading queries:** a search for a section's topic should find that section.
3. **Deep queries:** a sentence taken from *past* the 256-token cutoff of each long clause should still
   find its clause. This is what splitting fixes; it is compared against embedding whole clauses.


In [6]:

stored = vector_db.get(include=["metadatas", "documents"])
heading_tests = {
    "Safety requirements": {"swehb-3.2.1.4-guidance", "npr-3.7.3-requirement"},
    "Interface requirements": {"swehb-3.2.3-guidance"},
    "bi-directional traceability between requirements": {"npr-3.12.1-requirement", "npr-3.12.1-note"},
    "operator override of software functions requires two independent actions": {"npr-3.7.3-requirement", "swehb-3.2.1.4-guidance"},
    "how requirements will be verified: demonstration, testing, analysis, inspection": {"swehb-1.c-guidance"},
}
heading_hits = {q: bool(want & {c for c, _, _ in retrieve(q)}) for q, want in heading_tests.items()}

# one sentence from well past the cutoff in each long clause
deep = []
for _, row in clauses[too_long].iterrows():
    position = 0
    for s in sentences(row["text"]):
        position += n_tokens(s)
        if position > LIMIT + 60 and 10 <= len(s.split()) <= 35:
            deep.append((row["chunk_id"], s))
            break
deep_hits = sum(cid in {c for c, _, _ in retrieve(q)} for cid, q in deep)

# the same deep queries against whole, unsplit clauses (what the model sees without splitting)
whole = Chroma.from_documents(
    [Document(page_content=r["text"], metadata={"chunk_id": r["chunk_id"]}) for _, r in clauses.iterrows()],
    embeddings, collection_name="whole_clauses_comparison")
whole_top3 = lambda q: [d.metadata["chunk_id"] for d in whole.similarity_search(q, k=3)]
whole_hits = sum(cid in whole_top3(q) for cid, q in deep)
whole.delete_collection()

checks = {
    "every piece fits the model's token limit":
        all(n_tokens(d) <= LIMIT for d in stored["documents"]),
    "all 62 clauses stored":
        {m["chunk_id"] for m in stored["metadatas"]} == set(clauses["chunk_id"]),
    "citation metadata on every piece":
        all(set(META) <= set(m) for m in stored["metadatas"]),
    "no 'text:' prefix in stored text":
        not any(d.startswith("text:") for d in stored["documents"]),
    "heading queries find their section":
        all(heading_hits.values()),
    f"deep queries find their clause (split {deep_hits}/{len(deep)} vs whole {whole_hits}/{len(deep)})":
        deep_hits >= 0.85 * len(deep) and deep_hits > whole_hits,
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values()), "vector store check failed"


PASS every piece fits the model's token limit
PASS all 62 clauses stored
PASS citation metadata on every piece
PASS no 'text:' prefix in stored text
PASS heading queries find their section
PASS deep queries find their clause (split 32/35 vs whole 20/35)
